# 09 — Parallel Task Runner (Amazon FAR style)

Run a batch of independent tasks, capturing each result or exception. Parts 1-3 build the core
(concurrency at Part 3); Parts 4-5 are stretch tasks (retries/fail-fast, then a process pool). Fill
stubs, run each test cell, peek at the collapsed `ref_` solutions only after trying.

A "task" is a `(fn, args_tuple)` pair. Results come back as **Futures** in the same order as the input.

---

## Part 1 — Future

`Future` is a one-shot result box: `set_result(v)`, `set_exception(e)`, `result(timeout=None)` (blocks
until set, re-raises a stored exception, raises `TimeoutError` if the wait elapses), and `done()`.

**Lock down:** back it with a `threading.Event`; `result()` must work whether set before or after the
caller waits; setting twice is undefined (don't worry about it).

In [ ]:
import threading


class Future:
    def __init__(self):
        raise NotImplementedError

    def set_result(self, value):
        raise NotImplementedError

    def set_exception(self, exc):
        raise NotImplementedError

    def result(self, timeout=None):
        raise NotImplementedError

    def done(self):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    import time
    f = Future(); f.set_result(42)
    assert f.done() and f.result() == 42
    g = Future(); g.set_exception(ValueError("x"))
    try:
        g.result()
    except ValueError:
        pass
    else:
        raise AssertionError("expected ValueError")
    h = Future()
    threading.Thread(target=lambda: (time.sleep(0.02), h.set_result("late"))).start()
    assert h.result(timeout=2) == "late"            # blocks until another thread sets it
    try:
        Future().result(timeout=0.01)
    except TimeoutError:
        pass
    else:
        raise AssertionError("expected TimeoutError")
    print("Part 1 OK")

_t1()

## Part 2 — Sequential runner

`run_sequential(tasks) -> list[Future]`: run each `(fn, args)` in order and return a Future per task,
resolved with the result or, **if the task raised, the captured exception** (one bad task must not
abort the rest).

**Lock down:** failures are isolated into their Future; the returned list lines up with the input.

In [ ]:
def run_sequential(tasks):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    def sq(x):
        return x * x

    def boom():
        raise RuntimeError("nope")

    futs = run_sequential([(sq, (3,)), (boom, ()), (sq, (4,))])
    assert futs[0].result() == 9 and futs[2].result() == 16
    try:
        futs[1].result()
    except RuntimeError:
        pass
    else:
        raise AssertionError("expected the captured RuntimeError")
    print("Part 2 OK")

_t2()

## Part 3 — Parallel runner (worker pool)

`run_parallel(tasks, num_workers) -> list[Future]`: same semantics as Part 2, but executed across a
**bounded pool of `num_workers` threads**. Result order must still match the input order.

**Discuss:** dispatch tasks (tagged with their index) onto a `queue.Queue`, one sentinel per worker to
shut down; each worker resolves its task's Future; join the workers. Why tagging by index preserves
order even though completion order is nondeterministic.

In [ ]:
import queue


def run_parallel(tasks, num_workers):
    raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    def sq(x):
        return x * x

    N = 500
    futs = run_parallel([(sq, (i,)) for i in range(N)], num_workers=8)
    assert [f.result() for f in futs] == [i * i for i in range(N)]   # order preserved
    # exceptions are isolated per task
    def boom(i):
        if i % 7 == 0:
            raise ValueError(i)
        return i

    futs2 = run_parallel([(boom, (i,)) for i in range(50)], num_workers=4)
    for i, f in enumerate(futs2):
        if i % 7 == 0:
            try:
                f.result()
            except ValueError:
                pass
            else:
                raise AssertionError(i)
        else:
            assert f.result() == i
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Retries & fail-fast

`run_with_retries(tasks, num_workers, retries) -> list[Future]`: like `run_parallel`, but re-run a
**failing** task up to `retries` extra times before giving up (the Future then holds the last
exception). Transient failures should succeed on a later attempt.

**Discuss:** idempotency (only safe to retry side-effect-free tasks), exponential backoff + jitter in
real systems, and why a per-task *timeout* can't truly cancel a running Python thread (you can stop
*waiting*, but the thread keeps going) — name that limitation.

In [ ]:
def run_with_retries(tasks, num_workers, retries):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    class Flaky:
        def __init__(self, fail_times):
            self.fail_times = fail_times
            self.calls = 0
            self.lock = threading.Lock()

        def __call__(self):
            with self.lock:
                self.calls += 1
                c = self.calls
            if c <= self.fail_times:
                raise ValueError("transient")
            return "ok"

    futs = run_with_retries([(Flaky(2), ())], num_workers=2, retries=2)
    assert futs[0].result() == "ok"                 # succeeds on the 3rd attempt
    futs2 = run_with_retries([(Flaky(5), ())], num_workers=2, retries=1)
    try:
        futs2[0].result()
    except ValueError:
        pass
    else:
        raise AssertionError("should still fail after exhausting retries")
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Process pool for CPU-bound work

`run_multiprocess(items, num_procs) -> list`: apply a CPU-bound job to each item across processes with
`ProcessPoolExecutor`, preserving order. The worker is `threadpool_workers.cpu_task`.

**Discuss:** threads (Parts 1-4) are right for I/O/coordination but the GIL serializes CPU work — so
CPU-bound batches go to processes; pickling cost; chunking; and that a real executor offers both
thread and process backends behind one `submit` API.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import threadpool_workers


def run_multiprocess(items, num_procs):
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    items = [0, 1, 10, 100, 50]
    expected = [sum(i * i for i in range(n)) for n in items]
    assert run_multiprocess(items, num_procs=3) == expected
    print("Part 5 OK")

_t5()

## Likely follow-ups
- A unified `Executor` with `submit`/`map`/`shutdown(wait=)` and both thread and process backends.
- `as_completed` (yield futures as they finish) vs ordered results; cancellation of not-yet-started tasks.
- Bounded in-flight work / backpressure when tasks are produced faster than consumed.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
import queue
from concurrent.futures import ProcessPoolExecutor
import threadpool_workers


class RefFuture:
    def __init__(self):
        self._event = threading.Event()
        self._value = None
        self._exc = None

    def set_result(self, value):
        self._value = value
        self._event.set()

    def set_exception(self, exc):
        self._exc = exc
        self._event.set()

    def result(self, timeout=None):
        if not self._event.wait(timeout):
            raise TimeoutError()
        if self._exc is not None:
            raise self._exc
        return self._value

    def done(self):
        return self._event.is_set()


def ref_run_sequential(tasks):
    futs = []
    for fn, args in tasks:
        f = RefFuture()
        try:
            f.set_result(fn(*args))
        except Exception as e:        # noqa: BLE001 — isolate the failure into the Future
            f.set_exception(e)
        futs.append(f)
    return futs


def _drain_pool(tasks, num_workers, run_one):
    futs = [RefFuture() for _ in tasks]
    q = queue.Queue()
    for idx, (fn, args) in enumerate(tasks):
        q.put((idx, fn, args))
    for _ in range(num_workers):
        q.put(None)

    def worker():
        while True:
            item = q.get()
            if item is None:
                return
            idx, fn, args = item
            run_one(futs[idx], fn, args)

    threads = [threading.Thread(target=worker) for _ in range(num_workers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return futs


def ref_run_parallel(tasks, num_workers):
    def run_one(fut, fn, args):
        try:
            fut.set_result(fn(*args))
        except Exception as e:        # noqa: BLE001
            fut.set_exception(e)
    return _drain_pool(tasks, num_workers, run_one)


def ref_run_with_retries(tasks, num_workers, retries):
    def run_one(fut, fn, args):
        last = None
        for _ in range(retries + 1):
            try:
                fut.set_result(fn(*args))
                return
            except Exception as e:    # noqa: BLE001
                last = e
        fut.set_exception(last)
    return _drain_pool(tasks, num_workers, run_one)


def ref_run_multiprocess(items, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return list(ex.map(threadpool_workers.cpu_task, items))


_f = RefFuture(); _f.set_result(1); assert _f.result() == 1 and _f.done()
assert [x.result() for x in ref_run_sequential([(abs, (-3,)), (abs, (4,))])] == [3, 4]
assert [x.result() for x in ref_run_parallel([(abs, (-i,)) for i in range(20)], 4)] == list(range(20))
assert ref_run_multiprocess([0, 1, 5], 2) == [0, 0, 30]
print("reference solutions OK")